# MoCha 2026 -- Gait Severity Classification (v1 training pipeline)

Trains a custom classification head on top of a frozen, MoCha-organizer-provided
motion-text encoder to predict MDS-UPDRS gait severity (classes 0-3) from
canonicalized SMPL walking sequences, for the MoCha 2026 @ ECCV challenge.

This notebook reflects the exact pipeline used for the first submitted
version of the model. See `README.md` in
this repository for full technical documentation.

In [ ]:
# Cell 1: Install git-lfs and clone
!sudo apt-get install -y git-lfs -q
!git lfs install
!git clone https://github.com/TaatiTeam/MoCha_baseline_bundle /content/MoCha_baseline_bundle
%cd /content/MoCha_baseline_bundle
!git lfs pull
# Verify the real weights downloaded (should NOT be 133 bytes)
import os
for f in ['weights/backbone/motion_encoder_finetuned.pth',
          'weights/backbone/text_mot_match/model/finest.tar',
          'weights/classifier/classifier.pth',
          'weights/smpl/SMPL_NEUTRAL.pkl']:
    sz = os.path.getsize(f)
    print(f'{f}: {sz/1e6:.1f} MB  {"OK" if sz > 10000 else "FAIL - still LFS pointer"}')

In [ ]:
# Cell 2
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, '/content/MoCha_baseline_bundle')
import os, pickle, warnings
import numpy as np
from collections import Counter
warnings.filterwarnings('ignore')
DATA_DIR = '/content/drive/MyDrive/MoCha2026/care_pd_pickles'
LABELED_FILES = {
    '3DGait':   '3DGait_canonical.pkl',
    'BMCLab':   'BMCLab_canonical.pkl',
    'PD-GaM':   'PD-GaM_canonical.pkl',
    'T-SDU-PD': 'T-SDU-PD_canonical.pkl',
}
datasets = {}
for name, fname in LABELED_FILES.items():
    with open(os.path.join(DATA_DIR, fname), 'rb') as f:
        datasets[name] = pickle.load(f)
    n = sum(len(w) for w in datasets[name].values())
    print(f'  {name}: {len(datasets[name])} subjects, {n} walks')

In [ ]:
# Cell 3: Load the frozen baseline encoder
import torch
from pathlib import Path
from submission.baseline_model import load_config
from submission.preprocess import MotionPreprocessor
from model.t2m_eval_wrapper import EvaluatorModelWrapper
from utils.get_opt import get_opt
cfg = load_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
preprocessor = MotionPreprocessor(
    smpl_model_path=cfg['smpl_model_path'],
    normalization_dir=cfg['normalization_dir'],
    device=device,
    sequence_len=int(cfg['sequence_len']),
    target_fps=cfg.get('target_fps'),
    apply_slope_correction=False,
)
opt = get_opt(Path(cfg['opt_file_path']), device)
opt.checkpoints_dir = str(Path(cfg['checkpoints_dir']))
eval_wrapper = EvaluatorModelWrapper(opt)
# Load the FINETUNED encoder weights (overwrites the base checkpoint)
state = torch.load(cfg['motion_encoder_checkpoint_path'], map_location=device, weights_only=True)
eval_wrapper.motion_encoder.load_state_dict(state)
eval_wrapper.motion_encoder.eval()
eval_wrapper.movement_encoder.eval()
print('Finetuned encoder loaded.')

In [ ]:
# Cell 4: Extract embeddings for all labeled walks + save to Drive
from tqdm import tqdm
EMBED_CACHE = '/content/drive/MyDrive/MoCha2026/embeddings_v1.pkl'
if os.path.exists(EMBED_CACHE):
    print(f'Loading cached embeddings from {EMBED_CACHE}')
    with open(EMBED_CACHE, 'rb') as f:
        embed_data = pickle.load(f)
else:
    print('Extracting embeddings (this takes ~10-15 min)...')
    embed_data = []  # list of dicts
    for ds_name, ds_dict in datasets.items():
        for subj_id, walks in tqdm(ds_dict.items(), desc=ds_name):
            for walk_id, entry in walks.items():
                updrs = entry.get('UPDRS_GAIT')
                if updrs is None:
                    continue
                try:
                    motion, length = preprocessor(entry)
                    motion_t = torch.as_tensor(motion[None], dtype=torch.float32, device=device)
                    length_t = torch.as_tensor([length], dtype=torch.long, device=device)
                    with torch.no_grad():
                        emb = eval_wrapper.get_motion_embeddings_ordered(motion_t, length_t)
                    embed_data.append({
                        'embedding': emb.squeeze(0).cpu().numpy().astype(np.float32),
                        'updrs': int(updrs),
                        'dataset': ds_name,
                        'subject_id': str(subj_id),
                        'walk_id': str(walk_id),
                    })
                except Exception as e:
                    print(f'  Skipped {ds_name}/{subj_id}/{walk_id}: {e}')
    with open(EMBED_CACHE, 'wb') as f:
        pickle.dump(embed_data, f)
    print(f'Saved {len(embed_data)} embeddings to Drive.')
print(f'Total embeddings: {len(embed_data)}')
print(f'Class dist: {Counter(d["updrs"] for d in embed_data)}')
print(f'Embedding shape: {embed_data[0]["embedding"].shape}')

In [ ]:
# Cell 5: Build arrays
import numpy as np
X = np.array([d['embedding'] for d in embed_data], dtype=np.float32)
y = np.array([d['updrs'] for d in embed_data], dtype=np.int32)
groups = np.array([d['dataset'] for d in embed_data])
SITES = sorted(np.unique(groups).tolist())
class_counts = Counter(y)
n_total = len(y)
# Strongly upweight class 3 (45 samples) and class 2 (593 samples)
class_weights_np = np.array([
    (n_total / (4 * class_counts[c])) ** 1.2 for c in range(4)
], dtype=np.float32)
print('Class weights:', np.round(class_weights_np, 2))
print('Shapes:', X.shape, y.shape, 'Sites:', SITES)

In [ ]:
# Cell 6: Define the improved classifier head + LOSO training

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, cohen_kappa_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

class ImprovedClassifier(nn.Module):
    """Wider + deeper than baseline (512->256->128->64->4), with LayerNorm.
    Same interface as baseline MotionClassifier so it's a drop-in replacement."""
    def __init__(self, input_dim=512, num_classes=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )
        # Ordinal binary heads (for the 3 class boundaries)
        self.head_b1 = nn.Linear(64, 1)  # P(y >= 1)
        self.head_b2 = nn.Linear(64, 1)  # P(y >= 2)
        self.head_b3 = nn.Linear(64, 1)  # P(y >= 3)

    def forward(self, x):
        h = x
        for layer in self.fc[:-1]:  # all but final linear
            h = layer(h)
        logits = self.fc[-1](h)
        # Binary ordinal heads share the same 64-dim rep
        b1 = self.head_b1(h).squeeze(-1)
        b2 = self.head_b2(h).squeeze(-1)
        b3 = self.head_b3(h).squeeze(-1)
        return logits, b1, b2, b3


def ordinal_loss(logits, b1, b2, b3, targets, class_weights_tensor):
    """Main CE loss + ordinal binary BCE losses, class-weighted."""
    ce = F.cross_entropy(logits, targets, weight=class_weights_tensor)
    # Binary targets for each boundary
    t1 = (targets >= 1).float()
    t2 = (targets >= 2).float()
    t3 = (targets >= 3).float()
    # Per-sample weighting: upweight class-3 samples more
    sw = class_weights_tensor[targets]
    bce1 = F.binary_cross_entropy_with_logits(b1, t1, weight=sw)
    bce2 = F.binary_cross_entropy_with_logits(b2, t2, weight=sw)
    bce3 = F.binary_cross_entropy_with_logits(b3, t3, weight=sw)
    return ce + 0.3 * (bce1 + bce2 + bce3)


def train_fold(X_tr, y_tr, X_val, y_val, class_w, device, n_epochs=80, lr=3e-4):
    model = ImprovedClassifier(512, 4).to(device)
    cw = torch.tensor(class_w, device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)

    X_tr_t = torch.tensor(X_tr, device=device)
    y_tr_t = torch.tensor(y_tr, dtype=torch.long, device=device)
    X_val_t = torch.tensor(X_val, device=device)

    best_f1, best_state = 0.0, None
    for epoch in range(n_epochs):
        model.train()
        # Shuffle
        perm = torch.randperm(len(X_tr_t))
        for start in range(0, len(perm), 64):
            idx = perm[start:start+64]
            xb, yb = X_tr_t[idx], y_tr_t[idx]
            # Embedding-space augmentation: Gaussian noise + mixup
            if epoch < n_epochs * 0.7:  # augment for first 70% of training
                noise = torch.randn_like(xb) * 0.05
                xb = xb + noise
            opt.zero_grad()
            logits, b1, b2, b3 = model(xb)
            loss = ordinal_loss(logits, b1, b2, b3, yb, cw)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                logits_val, _, _, _ = model(X_val_t)
                preds = logits_val.argmax(1).cpu().numpy()
            f1 = f1_score(y_val, preds, average='macro', zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_state = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, best_f1


device_train = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

oof_preds, oof_true = [], []
loso_results = []

for holdout_site in SITES:
    print(f'\n=== Fold: holdout = {holdout_site} ===')
    tr_mask = groups != holdout_site
    val_mask = groups == holdout_site

    X_tr, y_tr = X[tr_mask], y[tr_mask]
    X_val, y_val = X[val_mask], y[val_mask]
    print(f'  Train: {len(X_tr)} | Val: {len(X_val)} | {Counter(y_val)}')

    # SMOTE on 512-dim embeddings
    n_maj = Counter(y_tr).most_common(1)[0][1]
    strat = {c: max(cnt, n_maj//5) for c, cnt in Counter(y_tr).items()}
    try:
        k_nn = min(5, max(2, Counter(y_tr)[3]-1))
        sm = SMOTE(sampling_strategy=strat, k_neighbors=k_nn, random_state=42)
        X_res, y_res = sm.fit_resample(X_tr, y_tr)
        print(f'  After SMOTE: {Counter(y_res)}')
    except Exception as e:
        print(f'  SMOTE skipped: {e}'); X_res, y_res = X_tr, y_tr

    model, best_f1 = train_fold(X_res, y_res, X_val, y_val, class_weights_np, device_train)
    print(f'  Best val macro F1: {best_f1:.4f}')

    model.eval()
    with torch.no_grad():
        logits, _, _, _ = model(torch.tensor(X_val, device=device_train))
        preds = logits.argmax(1).cpu().numpy()

    f1 = f1_score(y_val, preds, average='macro', zero_division=0)
    qwk = cohen_kappa_score(y_val, preds, weights='quadratic')
    print(f'  Final val F1: {f1:.4f} | QWK: {qwk:.4f}')
    oof_preds.extend(preds.tolist())
    oof_true.extend(y_val.tolist())
    loso_results.append({'site': holdout_site, 'f1': f1, 'qwk': qwk})

# Summary
print('\n=== LOSO Summary ===')
for r in loso_results:
    print(f"  {r['site']:12s}: F1={r['f1']:.4f}  QWK={r['qwk']:.4f}")
print(f"\nMean LOSO F1: {np.mean([r['f1'] for r in loso_results]):.4f}")
print(f'\nPooled OOF F1: {f1_score(oof_true, oof_preds, average="macro", zero_division=0):.4f}')
print(f'Pooled OOF QWK: {cohen_kappa_score(oof_true, oof_preds, weights="quadratic"):.4f}')
print()
print(classification_report(oof_true, oof_preds, zero_division=0))

In [ ]:
# Cell 7: Final training on all data
print('Training final classifier on all data...')
# SMOTE on full dataset
n_maj_all = Counter(y).most_common(1)[0][1]
strat_all = {c: max(cnt, n_maj_all//5) for c, cnt in Counter(y).items()}
sm_final = SMOTE(sampling_strategy=strat_all, k_neighbors=5, random_state=42)
X_all_res, y_all_res = sm_final.fit_resample(X, y)
print(f'Final training dist: {Counter(y_all_res)}')
# Train longer on full data (more epochs since no early stopping needed)
final_model = ImprovedClassifier(512, 4).to(device_train)
cw = torch.tensor(class_weights_np, device=device_train)
opt_final = torch.optim.AdamW(final_model.parameters(), lr=3e-4, weight_decay=1e-3)
sched_final = torch.optim.lr_scheduler.CosineAnnealingLR(opt_final, T_max=120)
X_all_t = torch.tensor(X_all_res, dtype=torch.float32, device=device_train)
y_all_t = torch.tensor(y_all_res, dtype=torch.long, device=device_train)
for epoch in range(120):
    final_model.train()
    perm = torch.randperm(len(X_all_t))
    total_loss = 0
    for start in range(0, len(perm), 64):
        idx = perm[start:start+64]
        xb, yb = X_all_t[idx], y_all_t[idx]
        if epoch < 84:
            xb = xb + torch.randn_like(xb) * 0.05
        opt_final.zero_grad()
        logits, b1, b2, b3 = final_model(xb)
        loss = ordinal_loss(logits, b1, b2, b3, yb, cw)
        loss.backward()
        nn.utils.clip_grad_norm_(final_model.parameters(), 1.0)
        opt_final.step()
        total_loss += loss.item()
    sched_final.step()
    if (epoch+1) % 30 == 0:
        print(f'  Epoch {epoch+1}/120, loss={total_loss:.3f}')
print('Final model trained.')
# Save just the classifier weights (drop the ordinal heads -- baseline run.py only uses self.fc)
# We save the FULL state_dict, but the submission needs to load into MotionClassifier
# Solution: save only the 'fc' weights that match baseline's MotionClassifier interface
# BUT keep our improved architecture -- we need to update baseline_model.py too
WEIGHTS_DIR = '/content/drive/MyDrive/MoCha2026/head_v1'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
torch.save(final_model.state_dict(), f'{WEIGHTS_DIR}/classifier_improved.pth')
print(f'Saved to {WEIGHTS_DIR}/classifier_improved.pth')

In [ ]:
UPDATED_MODEL_JSON = "\"\\\"\\\"\\\"Baseline model used by the MoCha sample submission -- improved classifier head.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any, Mapping\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn as nn\\n\\nfrom model.t2m_eval_wrapper import EvaluatorModelWrapper\\nfrom submission.preprocess import MotionPreprocessor\\nfrom utils.get_opt import get_opt\\n\\n\\nROOT = Path(__file__).resolve().parents[1]\\n\\n\\nclass ImprovedClassifier(nn.Module):\\n    \\\"\\\"\\\"Improved head: 512->256->128->64->4 with LayerNorm, ordinal binary heads.\\\"\\\"\\\"\\n    def __init__(self, input_dim: int = 512, num_classes: int = 4):\\n        super().__init__()\\n        self.fc = nn.Sequential(\\n            nn.Linear(input_dim, 256),\\n            nn.LayerNorm(256),\\n            nn.ReLU(),\\n            nn.Dropout(0.4),\\n            nn.Linear(256, 128),\\n            nn.LayerNorm(128),\\n            nn.ReLU(),\\n            nn.Dropout(0.3),\\n            nn.Linear(128, 64),\\n            nn.ReLU(),\\n            nn.Linear(64, num_classes),\\n        )\\n        self.head_b1 = nn.Linear(64, 1)\\n        self.head_b2 = nn.Linear(64, 1)\\n        self.head_b3 = nn.Linear(64, 1)\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        \\\"\\\"\\\"Returns class logits only (same interface as baseline MotionClassifier).\\\"\\\"\\\"\\n        h = x\\n        for layer in self.fc[:-1]:\\n            h = layer(h)\\n        return self.fc[-1](h)\\n\\n\\ndef load_config() -> dict[str, Any]:\\n    with (ROOT / \\\"weights\\\" / \\\"config.json\\\").open(\\\"r\\\", encoding=\\\"utf-8\\\") as f:\\n        cfg = json.load(f)\\n    for key in (\\\"classifier_checkpoint_path\\\",\\\"motion_encoder_checkpoint_path\\\",\\n                \\\"opt_file_path\\\",\\\"checkpoints_dir\\\",\\\"normalization_dir\\\",\\\"smpl_model_path\\\"):\\n        cfg[key] = (ROOT / cfg[key]).resolve()\\n    return cfg\\n\\n\\ndef choose_device() -> torch.device:\\n    return torch.device(\\\"cuda\\\" if torch.cuda.is_available() else \\\"cpu\\\")\\n\\n\\ndef torch_load(path: Path, device: torch.device):\\n    try:\\n        return torch.load(path, map_location=device, weights_only=True)\\n    except TypeError:\\n        return torch.load(path, map_location=device)\\n\\n\\nclass Model:\\n    def __init__(self):\\n        self.cfg = load_config()\\n        self.device = choose_device()\\n        self.preprocess = MotionPreprocessor(\\n            smpl_model_path=self.cfg[\\\"smpl_model_path\\\"],\\n            normalization_dir=self.cfg[\\\"normalization_dir\\\"],\\n            device=self.device,\\n            sequence_len=int(self.cfg[\\\"sequence_len\\\"]),\\n            target_fps=self.cfg.get(\\\"target_fps\\\"),\\n            apply_slope_correction=bool(self.cfg.get(\\\"apply_slope_correction\\\", False)),\\n        )\\n        self.eval_wrapper = self._load_motion_encoder()\\n        self.classifier = self._load_classifier()\\n\\n    def _load_motion_encoder(self) -> EvaluatorModelWrapper:\\n        opt = get_opt(Path(self.cfg[\\\"opt_file_path\\\"]), self.device)\\n        opt.checkpoints_dir = str(Path(self.cfg[\\\"checkpoints_dir\\\"]))\\n        wrapper = EvaluatorModelWrapper(opt)\\n        state = torch_load(Path(self.cfg[\\\"motion_encoder_checkpoint_path\\\"]), self.device)\\n        wrapper.motion_encoder.load_state_dict(state)\\n        wrapper.motion_encoder.eval()\\n        wrapper.movement_encoder.eval()\\n        return wrapper\\n\\n    def _load_classifier(self) -> ImprovedClassifier:\\n        model = ImprovedClassifier(512, int(self.cfg.get(\\\"num_classes\\\", 4))).to(self.device)\\n        model.load_state_dict(torch_load(Path(self.cfg[\\\"classifier_checkpoint_path\\\"]), self.device))\\n        model.eval()\\n        return model\\n\\n    def predict(self, sample: Mapping[str, Any]) -> int:\\n        motion, length = self.preprocess(sample)\\n        motion_t = torch.as_tensor(motion[None], dtype=torch.float32, device=self.device)\\n        length_t = torch.as_tensor([length], dtype=torch.long, device=self.device)\\n        with torch.no_grad():\\n            embedding = self.eval_wrapper.get_motion_embeddings_ordered(motion_t, length_t)\\n            logits = self.classifier(embedding)\\n        return int(torch.argmax(logits, dim=1).item())\\n\""

# Cell 8: Build submission zip
import shutil, zipfile, json as _json

SUB_DIR = '/content/mocha_submission_torch'

# Copy the entire baseline bundle as-is
shutil.copytree('/content/MoCha_baseline_bundle', SUB_DIR, dirs_exist_ok=True)

# 1. Replace the classifier weights
shutil.copy(
    f'{WEIGHTS_DIR}/classifier_improved.pth',
    f'{SUB_DIR}/weights/classifier/classifier.pth'
)

# 2. Write updated baseline_model.py that uses ImprovedClassifier
UPDATED_MODEL = _json.loads(UPDATED_MODEL_JSON)

with open(f'{SUB_DIR}/submission/baseline_model.py', 'w') as f:
    f.write(UPDATED_MODEL)

# 3. Zip it (from inside the directory so run.py is at root)
ZIP_PATH = '/content/mocha_submission_torch_v1.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(SUB_DIR):
        dirs[:] = [d for d in dirs if d not in ['__pycache__', '.git']]
        for file in files:
            if file.endswith('.pyc') or file.endswith('.DS_Store'):
                continue
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, SUB_DIR)
            zf.write(filepath, arcname)

print(f'Zip size: {os.path.getsize(ZIP_PATH)/1e6:.1f} MB')
print('Done.')
shutil.copy(ZIP_PATH, '/content/drive/MyDrive/MoCha2026/')